# Graph Enrichment

### Estimated time: 5-10 minutes

This lab closes the enrichment loop described in the paper. It takes the
schema-level suggestions from **7 - Augmentation Agent** and:

1. **Resolves** them into instance-level proposals — specific node-to-node
   relationships like `(Customer:C0001)-[INTERESTED_IN]->(Sector:Renewable Energy)`
2. **Filters** by confidence — HIGH auto-approves, MEDIUM flags for review, LOW rejects
3. **Writes back** approved proposals to Neo4j with provenance metadata
   (source document, extracted phrase, confidence, timestamp)
4. **Validates** the enrichment by querying for the new relationships

After this lab, the graph contains relationships that encode customer intent
alongside transactional data. Re-running **4 - Neo4j to Lakehouse** exports the
enriched graph, demonstrating the compounding loop: each cycle changes the graph,
which changes what the next cycle discovers.

### How Confidence Filtering Works

| Confidence | Action | Rationale |
|------------|--------|----------|
| **HIGH** | Auto-approve, write to graph | Document explicitly states the relationship |
| **MEDIUM** | Approve with flag, write to graph | Strongly implied but not explicit |
| **LOW** | Reject, report only | Loosely suggested, not enough evidence |

Both HIGH and MEDIUM proposals are written. LOW proposals are reported but not persisted.

### Prerequisites

- Completed **7 - Augmentation Agent** (results saved to volume)
- Neo4j instance running with data from **1 - Neo4j Import**
- Cluster library: `neo4j` Python driver, `databricks-openai`

## Step 1: Configuration

Enter your Supervisor Agent endpoint. The resolver calls it to cross-reference
suggested relationship types against actual graph data and documents.

In [ ]:
dbutils.widgets.text(
    "supervisor_endpoint", "",
    "Supervisor Agent Endpoint (e.g. mas-xxxxxxxx-endpoint)"
)

In [ ]:
%run ./Includes/config

In [ ]:
%run ./Includes/_lib/graph_enrichment

In [ ]:
SUPERVISOR_ENDPOINT = dbutils.widgets.get("supervisor_endpoint")
if not SUPERVISOR_ENDPOINT:
    raise ValueError(
        "Please enter the Supervisor Agent endpoint name in the widget above."
    )

_scope = CONFIG["secrets"]["scope_name"]
NEO4J_URL = dbutils.secrets.get(scope=_scope, key="url")
NEO4J_USER = dbutils.secrets.get(scope=_scope, key="username")
NEO4J_PASS = dbutils.secrets.get(scope=_scope, key="password")
VOLUME_PATH = dbutils.secrets.get(scope=_scope, key="volume_path")

print("Configuration:")
print(f"  Supervisor Endpoint: {SUPERVISOR_ENDPOINT}")
print(f"  Neo4j URL:           {NEO4J_URL}")
print(f"  Volume Path:         {VOLUME_PATH}")

## Step 2: Load Augmentation Results

Load the schema-level suggestions that **7 - Augmentation Agent** saved to the volume.

In [ ]:
import json
import os

results_path = os.path.join(VOLUME_PATH, "augmentation_results.json")

with open(results_path, "r") as f:
    augmentation_results = json.load(f)

suggested_relationships = augmentation_results.get("all_suggested_relationships", [])

print(f"[OK] Loaded results from: {results_path}")
print(f"     Total suggestions:         {augmentation_results.get('total_suggestions', 0)}")
print(f"     Suggested relationships:   {len(suggested_relationships)}")
print(f"     Suggested nodes:           {len(augmentation_results.get('all_suggested_nodes', []))}")
print(f"     Suggested attributes:      {len(augmentation_results.get('all_suggested_attributes', []))}")

if suggested_relationships:
    print("\n  Relationship types to resolve:")
    for rel in suggested_relationships:
        src = rel.get('source_label', '?')
        rtype = rel.get('relationship_type', '?')
        tgt = rel.get('target_label', '?')
        conf = rel.get('confidence', '?')
        print(f"    ({src})-[{rtype}]->({tgt}) [{conf}]")
else:
    print("\n  [WARN] No suggested relationships found. Run Lab 7 first.")

## Step 3: Resolve into Instance-Level Proposals

The resolver takes schema-level suggestions ("add INTERESTED_IN between Customer and
Sector") and calls the Supervisor Agent to produce concrete proposals with specific
node keys, confidence levels, and extracted evidence.

This step may take 1-3 minutes as the Supervisor Agent routes queries to both
Genie and the Knowledge Assistant.

In [ ]:
proposals = resolve_proposals(suggested_relationships, SUPERVISOR_ENDPOINT)

if proposals:
    print(f"\nProposals resolved:")
    for p in proposals:
        print(
            f"  ({p.source_node.label}:{p.source_node.key_value})"
            f"-[{p.relationship_type}]->"
            f"({p.target_node.label}:{p.target_node.key_value})"
            f"  [{p.confidence.value}]"
        )
        if p.extracted_phrase:
            print(f"    evidence: \"{p.extracted_phrase[:80]}...\"")
else:
    print("\n  No proposals resolved. Check the Supervisor Agent connection.")

## Step 4: Filter by Confidence

Sort proposals into decision buckets. HIGH and MEDIUM proceed to write-back.
LOW proposals are rejected and reported only.

In [ ]:
enrichment_result = filter_proposals(proposals)

writable = enrichment_result.approved + enrichment_result.flagged
print(f"\n  Proposals proceeding to write-back: {len(writable)}")

## Step 5: Write Back to Neo4j

Write approved proposals to Neo4j using idempotent MERGE statements.
Each enriched relationship gets provenance properties:

- `confidence` — HIGH or MEDIUM
- `source_document` — the document containing the evidence
- `extracted_phrase` — the quoted phrase supporting the proposal
- `enriched_at` — UTC timestamp of when the enrichment was written

Before writing, each proposal is validated:
- Both source and target nodes must exist in the graph
- All identifiers are checked for Cypher safety
- MERGE ensures running the same proposal twice updates rather than duplicates

In [ ]:
report = write_proposals(
    writable,
    neo4j_url=NEO4J_URL,
    neo4j_user=NEO4J_USER,
    neo4j_pass=NEO4J_PASS,
    dry_run=False,
)

print(f"\nSummary:")
print(f"  Created:  {report.created}")
print(f"  Updated:  {report.updated}")
print(f"  Skipped:  {len(report.skipped)}")
if report.skipped:
    print("\n  Skipped details:")
    for reason in report.skipped:
        print(f"    - {reason}")

## Step 6: Validate Enrichment

Query Neo4j for all enriched relationships (those with `enriched_at` property)
to confirm the write-back succeeded.

In [ ]:
validation = validate_enrichment(NEO4J_URL, NEO4J_USER, NEO4J_PASS)

## Step 7: Query the Enriched Graph

The enriched relationships are now queryable alongside transactional data.
Open Neo4j Browser and try these queries:

**Find all enriched relationships:**
```cypher
MATCH (src)-[r]->(tgt)
WHERE r.enriched_at IS NOT NULL
RETURN labels(src)[0] AS source, type(r) AS relationship,
       labels(tgt)[0] AS target, r.confidence, r.source_document
```

**The paper's key query — customers interested in renewable energy who hold none:**
```cypher
MATCH (c:Customer)-[r:INTERESTED_IN]->(s)
WHERE s.name CONTAINS 'Renewable' OR s.sectorId CONTAINS 'Renewable'
AND NOT EXISTS {
    MATCH (c)-[:HAS_ACCOUNT]->()-[:HAS_POSITION]->()-[:OF_SECURITY]->()-[:OF_COMPANY]->(co)
    WHERE co.sector CONTAINS 'Renewable'
}
RETURN c.first_name + ' ' + c.last_name AS customer,
       r.confidence, r.source_document, r.extracted_phrase
```

**Trace provenance for a specific enrichment:**
```cypher
MATCH (c:Customer)-[r]->()
WHERE r.enriched_at IS NOT NULL
RETURN c.first_name + ' ' + c.last_name AS customer,
       type(r) AS relationship,
       r.confidence, r.source_document,
       r.extracted_phrase, r.enriched_at
ORDER BY r.enriched_at DESC
```

## The Enrichment Loop

You have now completed one full pass through the enrichment loop:

1. **Extract** (Lab 4) — Neo4j graph data exported to Delta Lake
2. **Analyze** (Labs 5-6) — Agents compared structured data against documents
3. **Propose** (Lab 7) — DSPy analyzers produced schema-level suggestions
4. **Filter + Enrich** (Lab 8) — Proposals filtered by confidence and written back

To demonstrate the compounding behavior the paper describes, re-run
**4 - Neo4j to Lakehouse**. The exported Delta tables will now include the enriched
relationships. A second pass through Labs 7-8 would operate on a graph that contains
the new relationships, potentially discovering subtler patterns that only become
visible after the first enrichment expanded the graph's semantic layer.